# AEMO DUID Network Constraint Forecast Demo

This notebook is the final compact story for the `CATHROCK` 6h horizon demo: problem framing, public AEMO data sources, target discovery, DUID-to-constraint mapping, leakage-guarded horizon labels, Phase 3 benchmark results, API usage, and limitations.


## 0. Setup

Run from the repository root after installing `requirements.txt`. The notebook reads generated artifacts when they exist, so reviewers can inspect the packaged demo without rerunning every public-data download.


In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.aemo_constraint_demo.aemo_client import read_monthly_table
from src.aemo_constraint_demo.features import relevant_constraints_for_duid

DUID = "CATHROCK"
DAYS = 60
HORIZON_HOURS = 6
TARGET_COLUMN = "target_constrained_next_6h"
CACHE_DIR = REPO_ROOT / "data" / "cache"
MODEL_PATH = REPO_ROOT / "artifacts" / "cathrock_6h_best_model.joblib"
METRICS_PATH = REPO_ROOT / "artifacts" / "cathrock_6h_best_model_metrics.json"
BENCHMARK_PATH = REPO_ROOT / "artifacts" / "performance_benchmark.csv"
FEATURE_IMPORTANCE_PATH = REPO_ROOT / "artifacts" / "cathrock_6h_feature_importance.csv"
EXAMPLE_RESPONSE_PATH = REPO_ROOT / "artifacts" / "example_cathrock_6h_response.json"


## 1. Target discovery summary

The first `OAKEY1SF` attempt was useful as a smoke test but too sparse for a meaningful classifier. Phase 1.5 expanded target discovery across candidate DUIDs and forecast horizons, then selected `CATHROCK` because it had enough future-only 6h positives to demonstrate sparse-event modelling.


In [ ]:
scan_path = REPO_ROOT / "artifacts" / "duid_constraint_scan_expanded_top.csv"
if scan_path.exists():
    scan = pd.read_csv(scan_path)
    selected = scan[(scan["duid"] == DUID) & (scan["days"] == DAYS) & (scan["horizon_hours"] == HORIZON_HOURS)]
    display(selected.head(1)[[
        "duid",
        "days",
        "horizon_hours",
        "connection_point_id",
        "relevant_genconid_count",
        "positive_intervals",
        "positive_rate",
        "recommended",
    ]])
else:
    print("Run scripts/find_constrained_duids.py to regenerate target discovery artifacts.")


## 2. Map DUID -> connection point -> GENCONID

The public-data mapping is `DUID -> CONNECTIONPOINTID -> GENCONID`. The demo uses MMSDM standing-data tables such as `DUDETAIL`, `SPDCONNECTIONPOINTCONSTRAINT`, `GENCONSET`, and `GENCONSETINVOKE`, plus recent public Next Day Dispatch files for dispatch outcomes and constraint labels.


In [ ]:
dudetail = read_monthly_table("DUDETAIL", cache_dir=CACHE_DIR)
spd_cp = read_monthly_table("SPDCONNECTIONPOINTCONSTRAINT", cache_dir=CACHE_DIR)
# Planned/invoked constraint response support uses these standing-data tables in the training bundle.
genconset = read_monthly_table("GENCONSET", cache_dir=CACHE_DIR)
genconsetinvoke = read_monthly_table("GENCONSETINVOKE", cache_dir=CACHE_DIR)

relevant_genconids, connection_point, lhs_rows = relevant_constraints_for_duid(DUID, dudetail, spd_cp)
print({
    "duid": DUID,
    "connection_point": connection_point,
    "relevant_genconids": len(relevant_genconids),
})
lhs_rows.head()


## 3. Horizon label definition

For each interval `t`, `target_constrained_next_6h` is `1` when any relevant constraint binds in a future dispatch interval from `t+1` through `t+6h`. The current interval is excluded, and incomplete tail rows are dropped during training, so the label remains future-only.


In [ ]:
print(f"Target: {TARGET_COLUMN}")
print("Current interval is excluded from the future horizon label.")


## 4. Training command

Phase 3 benchmarks small, explainable model families and leakage-guarded feature sets. The selected bundle is logistic regression with time, weather, and dispatch-lag features.


In [ ]:
benchmark_command = """python scripts/benchmark_models.py \
  --duid CATHROCK \
  --days 60 \
  --horizon-hours 6 \
  --feature-sets time_only time_weather time_weather_dispatch_lag time_weather_constraint_lag \
  --model-types logistic_regression random_forest hist_gradient_boosting \
  --out artifacts/performance_benchmark.csv \
  --best-model-out artifacts/cathrock_6h_best_model.joblib \
  --report-out docs/reports/phase3_performance_report.md"""
print(benchmark_command)


## 5. Model metrics

The packaged best model is `logistic_regression` with `time_weather_dispatch_lag` features. The normal chronological holdout contains no positive `CATHROCK` 6h labels, so AP and ROC AUC there are `null`; the headline ranking metrics below come from event-window folds.


In [ ]:
if METRICS_PATH.exists():
    metrics = json.loads(METRICS_PATH.read_text())
    keys = [
        "duid",
        "days",
        "horizon_hours",
        "target_column",
        "model_type",
        "feature_set",
        "total_samples",
        "train_samples",
        "test_samples",
        "positive_rate",
        "test_positive_rate",
        "average_precision",
        "roc_auc",
        "precision_at_top_5_percent",
    ]
    display(pd.DataFrame([{key: metrics.get(key) for key in keys}]).T.rename(columns={0: "value"}))
else:
    print("Run scripts/benchmark_models.py to create artifacts/cathrock_6h_best_model_metrics.json.")


## 6. Event-window backtest summary

Phase 2.5 introduced `scripts/backtest_horizon_model.py` so rare-event folds train only on rows strictly before each validation window. Phase 3 reuses that event-window idea across model and feature-set benchmarks.


In [ ]:
if BENCHMARK_PATH.exists():
    benchmark = pd.read_csv(BENCHMARK_PATH)
    best = benchmark.sort_values(
        [
            "mean_average_precision",
            "mean_precision_at_top_5_percent",
            "mean_lift_at_top_5_percent",
            "mean_roc_auc",
        ],
        ascending=[False, False, False, False],
        na_position="last",
    ).head(1)
    display(best[[
        "model",
        "feature_set",
        "mean_average_precision",
        "mean_precision_at_top_5_percent",
        "mean_lift_at_top_5_percent",
        "mean_roc_auc",
        "evaluated_folds",
        "skipped_folds",
    ]])
else:
    print("Run scripts/benchmark_models.py to create artifacts/performance_benchmark.csv.")

if FEATURE_IMPORTANCE_PATH.exists():
    display(pd.read_csv(FEATURE_IMPORTANCE_PATH).head(10))


## 7. API query examples

The API infers `horizon_hours=6` from the best-model bundle when the query parameter is omitted. Explicit `horizon_hours=6` is also supported; mismatched horizons return a clear 400 error.


In [ ]:
print('uvicorn app:app --reload')
print('curl "http://127.0.0.1:8000/forecast/CATHROCK?model_path=artifacts/cathrock_6h_best_model.joblib&source=network"')
print('curl "http://127.0.0.1:8000/forecast/CATHROCK?model_path=artifacts/cathrock_6h_best_model.joblib&source=cache&end_date=2026-04-22"')

# The model is origin-anchored: one probability for the horizon starting at the latest
# observed dispatch interval, not a per-interval curve. See docs/constraint-forecasting.md, Phase 5.
if EXAMPLE_RESPONSE_PATH.exists():
    example = json.loads(EXAMPLE_RESPONSE_PATH.read_text())
    print({
        "duid": example["duid"],
        "horizon_hours": example["horizon_hours"],
        "model_type": example["model"]["model_type"],
        "as_of": example["as_of"],
        "data_lag_hours": example["data_lag_hours"],
        "probability": example["constraint_risk"]["probability"],
        "risk_level": example["constraint_risk"]["risk_level"],
        "features_imputed": example["feature_coverage"]["imputed"],
    })


## 8. Limitations

This is a transparent workflow demo, not a production-accurate forecaster. The event-window folds are few, the chronological holdout has no positive labels, and production work would need longer history, P5MIN/Predispatch inputs, outage context, regional demand and interconnector features, constraint taxonomy, calibration, and monitoring.
